# 06 — Rockfall Trajectories and Runout

**Short course:** *Geomorphological Hazards of Slopes* &nbsp;•&nbsp; University of Silesia in Katowice &nbsp;•&nbsp; 2 ECTS

*Lecturer: Ola Fredin*

---

Rockfall is what happens *after* slope failure has already occurred: a detached block, no longer part of a coherent landslide mass, races down a slope by falling, bouncing, rolling, and finally sliding to a stop. Predicting **where** that block stops — and what it hits on the way — is the central question of rockfall hazard zoning behind every alpine road, every fjord-side village, and every Carpathian valley town built on a talus apron.

This notebook covers the two complementary modelling approaches that every rockfall practitioner needs: the **shadow-angle / energy-line** method (empirical, defensible, used for first-cut hazard maps), and the **lumped-mass ballistic simulator** (mechanistic, parameter-hungry, used for site-specific design).


## About this notebook

**Learning objectives.** By the end of this notebook the student will be able to:

1. Distinguish the four motion modes of a falling block — falling, bouncing, rolling, sliding — and explain which dominates on which slope.
2. Apply Heim's fahrböschung (energy-line concept) and the Evans & Hungr (1993) α–β shadow-angle method to estimate maximum runout.
3. Implement and run a 2-D lumped-mass ballistic simulator with normal and tangential restitution coefficients.
4. Build a probabilistic runout map from a Monte-Carlo ensemble of trajectories with varied restitution.
5. Interpret the resulting runout-probability curve as the basis for a hazard-zoning map.

**Prerequisites.** Notebook 01 (landslide classification) for vocabulary, but otherwise self-contained. No prior soil-mechanics required — rockfall is a kinematic problem, not a soil-strength problem.

> **For your PowerPoint deck.** Four SVG figures land in `figures/`:
> - `shadow_angle_concept.svg` — energy-line and α/β definitions on a Norwegian fjord-slope profile
> - `rockfall_single_trajectory.svg` — one block, traced bounce by bounce
> - `rockfall_monte_carlo.svg` — 200 trajectories with varied restitution
> - `runout_probability.svg` — empirical CDF of stopping distance + hazard zones


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

from style import apply_style, COLORS, save_figure
apply_style()

from ipywidgets import interact, FloatSlider, IntSlider

G = 9.81  # m/s^2
np.random.seed(42)  # reproducible Monte Carlo


## 1. Four modes of motion

A falling block goes through a sequence of motion modes determined by the slope it is on:

- **Free fall.** Slope > 76°. The block is detached from the surface; only gravity acts. Vertical cliffs, headscarps.
- **Bouncing.** Slope 45–76°. The block intermittently contacts the surface, losing energy at each impact through normal restitution (compaction, sound, fracturing).
- **Rolling.** Slope 25–45°. Continuous contact; the block rotates and loses energy to surface deformation.
- **Sliding.** Slope < 25°. The block slides until friction halts it.

The transition angles are empirical and vary with block size and shape, but the sequence is reliable. A typical Norwegian fjord-slope sees all four in sequence: free fall off a 200-m cliff, bouncing down a steep talus, rolling across a gentler apron, sliding to rest on the beach.


## 2. The energy line and Heim's fahrböschung

Heim (1932) noticed that the **straight line from the source to the stopping point** of a rockfall makes a remarkably consistent angle with the horizontal across many events. He called this angle the *fahrböschung* (German: "travel slope"):

$$
\tan\alpha \;=\; \frac{H}{L},
\qquad (1)
$$

where $H$ is the total fall height from source to deposit and $L$ is the horizontal travel distance. For individual rockfalls in fresh, hard rock, $\alpha$ is typically 30–45°. For large rock avalanches (volumes > 10⁶ m³) $\alpha$ drops to 10–20° — the famous "long runout" of large failures.

The line itself is the **energy line**: it represents the height to which the block could climb at any horizontal position if all its kinetic energy were converted back to potential. The block cannot rise above it.

The energy-line concept gives an immediate, defensible, zero-parameter estimate of the maximum runout: given source elevation $H_0$ and a minimum fahrböschung $\alpha_{\min}$ (taken from a regional rockfall inventory, typically the lower envelope of observed events), the runout is

$$
L_{\max} \;=\; \frac{H_0}{\tan\alpha_{\min}}.
\qquad (2)
$$

Evans & Hungr (1993) formalised this in their **α–β method**: $\beta$ is the angle from the source to the upper edge of the existing talus (the topographic constraint), and $\alpha$ is the regression-derived lower envelope of the additional travel beyond the talus toe. For shallow rockfalls in the Coast Mountains of British Columbia they reported $\alpha_{\min}$ ≈ 27.5°, which is widely used as a default in hazard mapping.


In [ ]:
# Norwegian fjord-slope profile (synthetic but realistic for inner Møre fjords).
# Cliff -> steep talus -> gentle talus -> outwash -> fjord shore.
# We define a few key breakpoints, then densify with a monotonic spline so
# the slope is continuous everywhere (no artificial trapping at vertices).
_key_x = np.array([   0.0,   30.0,  120.0,  280.0,  500.0,  700.0,  900.0,  1300.0])
_key_y = np.array([ 600.0,  500.0,  300.0,  150.0,   80.0,   40.0,   15.0,     0.0])
try:
    from scipy.interpolate import PchipInterpolator
    _interp = PchipInterpolator(_key_x, _key_y)
    terrain_x = np.linspace(_key_x[0], _key_x[-1], 400)
    terrain_y = _interp(terrain_x)
except ImportError:
    terrain_x, terrain_y = _key_x, _key_y

source_x, source_y = 0.0, 600.0

# Energy lines at two fahrböschung angles
alpha_deg_default = 32.0
alpha_deg_min     = 27.5
x_extend = np.linspace(0, 1500, 200)
y_line_default = source_y - x_extend * np.tan(np.radians(alpha_deg_default))
y_line_min     = source_y - x_extend * np.tan(np.radians(alpha_deg_min))

L_default = source_y / np.tan(np.radians(alpha_deg_default))
L_min     = source_y / np.tan(np.radians(alpha_deg_min))

fig, ax = plt.subplots(figsize=(9.5, 5.5))

# Terrain
ax.fill_between(np.concatenate(([terrain_x[0]], terrain_x, [terrain_x[-1]])),
                np.concatenate(([0],            terrain_y, [0])),
                np.concatenate(([0],            np.zeros_like(terrain_y) - 50, [0])),
                color=COLORS["soil"], alpha=0.25)
ax.plot(terrain_x, terrain_y, color=COLORS["soil"], lw=2.5, label="ground profile")

# Source marker
ax.plot([source_x], [source_y], "o", ms=12, color=COLORS["fail"])
ax.annotate("source", (source_x, source_y), xytext=(8, 6), textcoords="offset points",
            color=COLORS["fail"], fontsize=12)

# Energy lines
ax.plot(x_extend, y_line_default, color=COLORS["accent"], lw=2.0, ls="--",
        label=fr"energy line  $\alpha$ = {alpha_deg_default:.1f}$^\circ$ (median rockfall)")
ax.plot(x_extend, y_line_min, color=COLORS["fail"], lw=2.0, ls=":",
        label=fr"shadow angle  $\alpha_{{\min}}$ = {alpha_deg_min:.1f}$^\circ$ (Evans & Hungr 1993)")

# Mark intersections with terrain
ax.axvline(L_default, color=COLORS["accent"], lw=0.8, alpha=0.5)
ax.axvline(L_min, color=COLORS["fail"], lw=0.8, alpha=0.5)
ax.annotate(fr"$L$ = {L_default:.0f} m", xy=(L_default, 0), xytext=(0, -20),
            textcoords="offset points", color=COLORS["accent"], fontsize=11, ha="center")
ax.annotate(fr"$L_{{\max}}$ = {L_min:.0f} m", xy=(L_min, 0), xytext=(0, -38),
            textcoords="offset points", color=COLORS["fail"], fontsize=11, ha="center")

ax.set_xlim(0, 1300); ax.set_ylim(-60, 700)
ax.set_xlabel("horizontal distance from cliff [m]")
ax.set_ylabel("elevation [m]")
ax.set_title(r"Heim's fahrböschung and the Evans & Hungr (1993) shadow angle on a fjord-slope")
ax.legend(loc="upper right")
save_figure(fig, "shadow_angle_concept")
plt.show()

print(f"Energy-line runout at alpha = {alpha_deg_default:.1f} deg:   L = {L_default:.0f} m")
print(f"Energy-line runout at alpha = {alpha_deg_min:.1f} deg:   L = {L_min:.0f} m (max expected)")


## 3. The lumped-mass ballistic simulator

The energy-line method is empirical: it tells you *where* a block stops, on average, but not *how* it got there. For site-specific design — protective fences, sheds, road tunnels — you need the trajectory itself, including bounce heights and impact energies. The standard approach is the **lumped-mass model**: treat the block as a point of mass $m$, integrate Newton's equations during flight, and apply restitution coefficients at each impact.

Between bounces the motion is simple projectile motion under gravity:

$$
\frac{d^2 \mathbf{x}}{dt^2} \;=\; -g\,\hat{\mathbf{y}}.
\qquad (3)
$$

At each impact with the terrain, the velocity is decomposed into components normal ($v_n$) and tangential ($v_t$) to the local slope, and each is multiplied by a **coefficient of restitution**:

$$
v_n' \;=\; -R_n\, v_n,
\qquad
v_t' \;=\; R_t\, v_t.
\qquad (4)
$$

Typical values (Pfeiffer & Bowen 1989; RocFall manual):

| surface | $R_n$ | $R_t$ |
|---|---|---|
| Fresh rock outcrop | 0.35 | 0.85 |
| Talus / coarse scree | 0.30 | 0.80 |
| Compact soil | 0.20 | 0.65 |
| Forest floor / litter | 0.10 | 0.45 |
| Snow / scree mix | 0.15 | 0.55 |

The block comes to rest when its speed drops below a small threshold (typically 0.5–1 m/s) on a slope shallower than the friction angle of the surface.


In [ ]:
def simulate_rockfall(x0, y0, vx0, vy0, terrain_x, terrain_y,
                     R_n=0.30, R_t=0.80, dt=0.01, v_min=0.7, max_steps=50000):
    """Simulate a 2-D rockfall trajectory over a piecewise-linear terrain.

    Returns the (x, y) path and an array of bounce locations.
    """
    def y_terrain(x):
        return np.interp(x, terrain_x, terrain_y)

    def slope_dy_dx(x):
        delta = 0.5  # finite-difference step (m)
        return (y_terrain(x + delta) - y_terrain(x - delta)) / (2 * delta)

    xs, ys = [x0], [y0]
    bounces = []
    x, y, vx, vy = x0, y0, vx0, vy0

    for _ in range(max_steps):
        # One Euler step
        x_new = x + vx * dt
        y_new = y + vy * dt
        vy_new = vy - G * dt

        if y_new <= y_terrain(x_new):
            # Snap to terrain at the new x
            y_impact = y_terrain(x_new)
            # Outward unit normal to terrain
            slope = slope_dy_dx(x_new)
            n_mag = np.sqrt(1.0 + slope * slope)
            nx, ny = -slope / n_mag, 1.0 / n_mag

            # Decompose pre-impact velocity (use the pre-step velocity)
            v_n = vx * nx + vy * ny
            v_tx = vx - v_n * nx
            v_ty = vy - v_n * ny

            # Restitution
            v_n_new = -R_n * v_n
            vx = R_t * v_tx + v_n_new * nx
            vy = R_t * v_ty + v_n_new * ny

            bounces.append((x_new, y_impact))

            speed = np.hypot(vx, vy)
            local_slope_deg = abs(np.degrees(np.arctan(slope)))
            # Predicted apex of next bounce above the terrain
            v_n_after = vx * nx + vy * ny
            bounce_height_apex = max(v_n_after, 0.0)**2 / (2.0 * G)

            # Three reasons to terminate:
            #   (a) slow speed on a gentle slope (block effectively at rest)
            #   (b) next bounce barely lifts off (block is rolling/sliding to a stop)
            #   (c) running off the end of the modelled terrain
            if ((speed < v_min and local_slope_deg < 25.0)
                or (bounce_height_apex < 0.05 and speed < 3.0)
                or x_new > terrain_x[-1] - 5.0):
                xs.append(x_new); ys.append(y_impact)
                break

            x, y = x_new, y_impact + 0.05
        else:
            x, y, vy = x_new, y_new, vy_new
        xs.append(x); ys.append(y)

    return np.array(xs), np.array(ys), np.array(bounces) if bounces else np.empty((0, 2))


# A single illustrative trajectory.
xs, ys, bnc = simulate_rockfall(
    x0=source_x, y0=source_y, vx0=2.0, vy0=0.0,
    terrain_x=terrain_x, terrain_y=terrain_y,
    R_n=0.32, R_t=0.82,
)
stop_x = xs[-1]

fig, ax = plt.subplots(figsize=(9.5, 5.5))
ax.fill_between(np.concatenate(([terrain_x[0]], terrain_x, [terrain_x[-1]])),
                np.concatenate(([0],            terrain_y, [0])),
                np.concatenate(([-50],          -50*np.ones_like(terrain_y), [-50])),
                color=COLORS["soil"], alpha=0.25)
ax.plot(terrain_x, terrain_y, color=COLORS["soil"], lw=2.5, label="ground profile")
ax.plot(xs, ys, color=COLORS["fail"], lw=1.5, alpha=0.8, label="block trajectory")
if len(bnc):
    ax.plot(bnc[:, 0], bnc[:, 1], "o", ms=5, color=COLORS["fail"], alpha=0.7,
            label=f"bounces  (n = {len(bnc)})")
ax.plot([source_x], [source_y], "o", ms=12, color="black", label="source")
ax.plot([stop_x], [ys[-1]], "X", ms=14, color=COLORS["safe"], label=f"rest at  x = {stop_x:.0f} m")

ax.set_xlim(0, 1300); ax.set_ylim(-50, 700)
ax.set_xlabel("horizontal distance from cliff [m]")
ax.set_ylabel("elevation [m]")
ax.set_title(fr"Rockfall trajectory over a Norwegian fjord-slope  ($R_n$ = 0.32, $R_t$ = 0.82)")
ax.legend(loc="upper right")
save_figure(fig, "rockfall_single_trajectory")
plt.show()

print(f"Block came to rest at x = {stop_x:.1f} m after {len(bnc)} bounces.")


## 4. Interactive trajectory exploration

The two restitution coefficients are by far the most important parameters in a lumped-mass simulation. Try the sliders below and observe:

- **Lower $R_n$** (forest floor, snow) → smaller bounces, shorter runout.
- **Lower $R_t$** (steep talus, friction-limited) → energy is bled off tangentially; rolling becomes sliding sooner.
- **Higher initial $v_{x0}$** (the block was launched, not merely released) → much greater runout, especially over the cliff face where the block clears the upper talus.

Realistic values for an alpine talus apron are $R_n$ = 0.25–0.35 and $R_t$ = 0.75–0.85. Values outside that range usually mean either a freshly fractured outcrop ($R_n$ → 0.4) or a heavily vegetated forest floor ($R_n$ → 0.05).


In [ ]:
def explore_trajectory(R_n=0.30, R_t=0.80, vx0=2.0):
    xs, ys, bnc = simulate_rockfall(
        x0=source_x, y0=source_y, vx0=vx0, vy0=0.0,
        terrain_x=terrain_x, terrain_y=terrain_y,
        R_n=R_n, R_t=R_t,
    )
    fig, ax = plt.subplots(figsize=(9.0, 4.6))
    ax.fill_between(terrain_x, terrain_y, -50, color=COLORS["soil"], alpha=0.25)
    ax.plot(terrain_x, terrain_y, color=COLORS["soil"], lw=2.0)
    ax.plot(xs, ys, color=COLORS["fail"], lw=1.4, alpha=0.85)
    if len(bnc):
        ax.plot(bnc[:, 0], bnc[:, 1], "o", ms=4, color=COLORS["fail"], alpha=0.7)
    ax.plot([xs[-1]], [ys[-1]], "X", ms=12, color=COLORS["safe"])
    ax.set_xlim(0, 1300); ax.set_ylim(-50, 700)
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
    ax.set_title(fr"$R_n$ = {R_n:.2f}, $R_t$ = {R_t:.2f}, $v_{{x0}}$ = {vx0:.1f} m/s "
                 fr"$\Rightarrow$ runout = {xs[-1]:.0f} m,  {len(bnc)} bounces")
    plt.show()


interact(
    explore_trajectory,
    R_n=FloatSlider(min=0.05, max=0.45, step=0.02, value=0.30, description=r"$R_n$"),
    R_t=FloatSlider(min=0.30, max=0.95, step=0.02, value=0.80, description=r"$R_t$"),
    vx0=FloatSlider(min=0.0,  max=10.0, step=0.5,  value=2.0,  description=r"$v_{x0}$ [m/s]"),
);


## 5. Monte-Carlo runout: from one trajectory to a hazard zone

A single deterministic trajectory is rarely useful for hazard mapping. The restitution coefficients are uncertain (they depend on block shape, impact geometry, surface roughness, moisture), and small differences propagate into large runout differences. The pragmatic approach is **Monte-Carlo simulation**: draw $R_n$ and $R_t$ from plausible distributions, run a few hundred trajectories, and turn the resulting cloud of stopping points into a probability of exceedance.

Below we use truncated normal distributions:
- $R_n \sim \mathcal{N}(0.30,\, 0.05)$, clipped to $[0.10, 0.45]$
- $R_t \sim \mathcal{N}(0.80,\, 0.05)$, clipped to $[0.55, 0.95]$


In [ ]:
def truncated_normal(mu, sigma, lo, hi, size):
    samples = np.random.normal(mu, sigma, size=size)
    return np.clip(samples, lo, hi)


N = 200
R_n_samples = truncated_normal(0.30, 0.05, 0.10, 0.45, N)
R_t_samples = truncated_normal(0.80, 0.05, 0.55, 0.95, N)

stopping_x = np.empty(N)
all_paths = []
for i in range(N):
    xs, ys, _ = simulate_rockfall(
        x0=source_x, y0=source_y, vx0=2.0, vy0=0.0,
        terrain_x=terrain_x, terrain_y=terrain_y,
        R_n=R_n_samples[i], R_t=R_t_samples[i],
    )
    stopping_x[i] = xs[-1]
    if i < 60:  # plot only a subset to keep the figure readable
        all_paths.append((xs, ys))

# Overlay plot
fig, ax = plt.subplots(figsize=(9.5, 5.5))
ax.fill_between(terrain_x, terrain_y, -50, color=COLORS["soil"], alpha=0.25)
ax.plot(terrain_x, terrain_y, color=COLORS["soil"], lw=2.5)
for xs, ys in all_paths:
    ax.plot(xs, ys, color=COLORS["fail"], lw=0.6, alpha=0.18)
ax.plot([source_x], [source_y], "o", ms=12, color="black", label="source")
ax.plot(stopping_x, np.interp(stopping_x, terrain_x, terrain_y),
        "X", ms=7, color=COLORS["safe"], alpha=0.5,
        label=f"stopping points  (N = {N})")
ax.set_xlim(0, 1300); ax.set_ylim(-50, 700)
ax.set_xlabel("horizontal distance from cliff [m]")
ax.set_ylabel("elevation [m]")
ax.set_title(f"Monte-Carlo rockfall ensemble  (N = {N} trajectories, varied $R_n, R_t$)")
ax.legend(loc="upper right")
save_figure(fig, "rockfall_monte_carlo")
plt.show()

print(f"Min / median / max runout: {stopping_x.min():.0f}, "
      f"{np.median(stopping_x):.0f}, {stopping_x.max():.0f}  m")
print(f"95-percentile runout:       {np.percentile(stopping_x, 95):.0f} m")


## 6. From a stopping-point cloud to hazard zones

The Monte-Carlo cloud collapses into a one-dimensional **runout exceedance curve**: the probability that a rockfall reaches at least a given horizontal distance. This is the foundation of every quantitative rockfall hazard map.

A common convention:

- **Red zone** (high hazard): probability of being reached > 10⁻². The 99-percentile runout marks its outer boundary.
- **Orange zone** (medium hazard): 10⁻⁴ < probability ≤ 10⁻². Buildings restricted; protection structures required.
- **Yellow zone** (low hazard): 10⁻⁵ < probability ≤ 10⁻⁴. Buildings allowed with reinforcement.

The thresholds are jurisdiction-specific (Norway, Switzerland, Italy all differ); the underlying logic is the same. Our 200-trajectory ensemble is too small for the orange/yellow boundaries — those need 10⁵ trajectories to resolve — but the red boundary is well constrained.


In [ ]:
# Empirical exceedance: P(L > x) = (# of trajectories that reached x) / N
sorted_L = np.sort(stopping_x)
exceedance = 1.0 - np.arange(N) / N  # at sorted_L[i], probability of exceeding is (N-i)/N

P50 = np.percentile(stopping_x, 50)
P95 = np.percentile(stopping_x, 95)
P99 = np.percentile(stopping_x, 99)

fig, ax = plt.subplots(figsize=(8.5, 5.0))
ax.step(sorted_L, exceedance, where="post", color=COLORS["fail"], lw=2.2)
ax.set_yscale("log")
ax.set_xlabel("horizontal distance from cliff [m]")
ax.set_ylabel(r"P(runout $\geq$ x)")
ax.set_title("Empirical runout-exceedance curve  —  basis for a hazard-zone map")

for label, val, colour in [("P50", P50, COLORS["accent"]),
                           ("P95", P95, COLORS["water"]),
                           ("P99", P99, COLORS["fail"])]:
    ax.axvline(val, color=colour, lw=1.2, ls="--")
    ax.text(val, 0.7, fr"  {label} = {val:.0f} m", color=colour, fontsize=11)

# Optional: shade hazard zones on the x-axis ribbon
ax.axhspan(1e-2, 1.0, alpha=0.10, color=COLORS["fail"])
ax.axhspan(1e-4, 1e-2, alpha=0.10, color=COLORS["accent"])
ax.set_ylim(1e-3, 1.0)
ax.set_xlim(0, max(sorted_L) * 1.05)
save_figure(fig, "runout_probability")
plt.show()

print(f"Median (P50)  runout: {P50:.0f} m")
print(f"P95           runout: {P95:.0f} m")
print(f"P99 (red zone): {P99:.0f} m")


## 7. Limitations

The lumped-mass model is a teaching tool with three big simplifications:

- **Block shape and rotation are ignored.** A platy block tumbles more chaotically than a cube; a spherical block runs further than either. Discrete-element codes (e.g. *Rockyfor3D*, *RAMMS::Rockfall*) treat shape explicitly and predict measurably different runouts.
- **Fragmentation is ignored.** Large blocks often shatter on first impact, distributing energy to smaller fragments that go further than the parent would have. Most operational simulators include a fragmentation submodel.
- **Vegetation and protection structures are not represented.** A mature forest cuts rockfall runout by tens of percent through repeated low-restitution impacts; a properly sited rockfall fence cuts it by an order of magnitude. Hazard maps that ignore mitigation overstate the risk to existing settlements.
- **2-D projection.** Real rockfalls deflect laterally. A 3-D simulation over a DEM is essential anywhere the source is more than ~30 m wide laterally.

For operational use, the open-source [Rockyfor3D](https://www.ecorisq.org) (Dorren 2003, Dorren et al. 2006) handles all four and is the most-cited 3-D rockfall code in European practice.


## Take-aways

- Rockfall passes through four motion modes (free fall, bouncing, rolling, sliding) determined by local slope angle.
- Heim's fahrböschung gives a zero-parameter first-cut estimate of maximum runout from a regional $\alpha_{\min}$ — typically 27–32° for individual rockfalls in hard rock.
- The lumped-mass ballistic model adds detail: bounce heights, impact energies, trajectory. The two most important parameters are the normal and tangential restitution coefficients.
- Hazard zoning is fundamentally probabilistic. A Monte-Carlo ensemble over plausible restitution distributions produces a runout-exceedance curve, from which red / orange / yellow zones are read at fixed probability thresholds.
- Operational codes (Rockyfor3D, RAMMS::Rockfall) extend the lumped-mass approach with block shape, fragmentation, vegetation, and 3-D topography. The teaching simulator in this notebook captures the right *logic* but should not be used for design.


## Questions for the exam

1. Define the fahrböschung angle $\alpha$ and explain why large rock avalanches systematically have smaller $\alpha$ than individual rockfalls.
2. A 200-m cliff has a regional shadow angle $\alpha_{\min}$ = 28°. What is the worst-case horizontal runout? A village sits 350 m from the cliff base at the foot of the talus apron. Is it inside the shadow zone?
3. Two slopes have identical geometry but different surface conditions: one is fresh talus ($R_n$ = 0.35, $R_t$ = 0.85), the other is a mature spruce forest ($R_n$ = 0.10, $R_t$ = 0.45). Predict, qualitatively, which gives a longer runout and why.
4. Why is a single deterministic rockfall trajectory of limited use for hazard mapping? What information does a Monte-Carlo ensemble add?


## References

- Heim, A. (1932). *Bergsturz und Menschenleben.* Vierteljahrsschrift der Naturforschenden Gesellschaft Zürich, 77, 1–218.
- Pfeiffer, T. J. & Bowen, T. D. (1989). *Computer simulation of rockfalls.* Bulletin of the Association of Engineering Geologists, 26(1), 135–146.
- Evans, S. G. & Hungr, O. (1993). *The assessment of rockfall hazard at the base of talus slopes.* Canadian Geotechnical Journal, 30(4), 620–636.
- Dorren, L. K. A. (2003). *A review of rockfall mechanics and modelling approaches.* Progress in Physical Geography, 27(1), 69–87.
- Blikra, L. H., Nilsen, B., Anda, E., Longva, O., & Braathen, A. (2006). *Rock-slope failures in Norway: typology, geological controls and hazard management.* Norsk Geologisk Tidsskrift, 86(3), 245–264.
